In [2]:
!pip install aisuite

In [3]:

import aisuite as ai


In [ ]:
key= ""

In [5]:
client = ai.Client(
    {
        "openai":{
            "timeout": 100,
            "api_key": key
        }
    }
)

MODEL = "openai:gpt-4o-mini"


In [6]:
messages = [
    {"role": "system", "content": "Rispondi come un assistente generico."},
    {"role": "user", "content": "Che ore sono?"},
]


In [11]:
response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.75
    )
print(response.choices[0].message.content)

Mi dispiace, ma non posso fornire l'ora attuale. Ti consiglio di controllare un orologio o il tuo dispositivo per avere l'ora esatta.


In [18]:
def orario():
  """questa funzione restituisce l'ora attuale"""
  import datetime
  ora= datetime.datetime.now().strftime("%H:%M")
  return {"ora":ora}

In [25]:
def orariov2():
  """questa funzione restituisce l'ora attuale e il giorno"""
  import datetime
  ora= datetime.datetime.now().strftime("%H:%M")
  giorno = datetime.datetime.now().strftime("%d-%m-%Y")
  return {"ora":ora, "giorno":giorno}

In [27]:
orariov2()

{'ora': '16:33', 'giorno': '20-11-2026'}

In [35]:
messages = [
    {"role": "system", "content": "Rispondi come un assistente generico. Se ti viene chiesta l'ora utiliza il tool orario. se non puoi darmi la risposta non inventarla. se ti viene chiesto quanti giorni mancano aduna certa data, utilizza come orario di partenza quello fornito dal tool orariov2"},
    {"role": "user", "content": "che tempo fa oggi?"},
]

In [36]:
response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        max_turns=3,
        temperature=0.75,
        tools=[orariov2]
    )
print(response.choices[0].message.content)

Oggi è il 20 novembre 2026 e l'ora attuale è 16:41. Per quanto riguarda il tempo, non posso fornire informazioni meteorologiche. Ti consiglio di controllare un servizio meteo locale o un'app per avere dettagli sul tempo attuale.


In [38]:
!pip install openmeteo_requests
!pip install requests_cache
!pip install retry_requests
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

In [39]:
def meteo():
  # Setup the Open-Meteo API client with cache and retry on error
  cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
  retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
  openmeteo = openmeteo_requests.Client(session = retry_session)

  # Make sure all required weather variables are listed here
  # The order of variables in hourly or daily is important to assign them correctly below
  url = "https://api.open-meteo.com/v1/forecast"
  params = {
    "latitude": 40.3548,
    "longitude": 18.1724,
    "current": ["temperature_2m", "rain", "cloud_cover", "relative_humidity_2m", "wind_speed_10m"],
    "timezone": "auto",
  }
  responses = openmeteo.weather_api(url, params = params)

  # Process first location. Add a for-loop for multiple locations or weather models
  response = responses[0]

  # Process current data. The order of variables needs to be the same as requested.
  current = response.Current()
  current_temperature_2m = current.Variables(0).Value()
  current_rain = current.Variables(1).Value()
  current_cloud_cover = current.Variables(2).Value()
  current_relative_humidity_2m = current.Variables(3).Value()
  current_wind_speed_10m = current.Variables(4).Value()
  return {"temperatura":current_temperature_2m, "pioggia":current_rain,
          "nuvole":current_cloud_cover,"umiditá":current_relative_humidity_2m,
          "vento":current_wind_speed_10m}

In [40]:
meteo()

{'temperatura': 17.989500045776367,
 'pioggia': 0.0,
 'nuvole': 51.0,
 'umiditá': 61.0,
 'vento': 14.372974395751953}

In [51]:
messages = [
    {"role": "system", "content": "Rispondi come un assistente generico. Se ti viene chiesta l'ora utiliza il tool orario. se ti viene chiesto quanti giorni mancano aduna certa data, utilizza come orario di partenza quello fornito dal tool orariov2. nel contesto del meteo per le domande relative alla nuvolositá, non usare le percentuali ma un'espressione piú popolare"},
    {"role": "user", "content": "quanto fa 11723,312 x 18,214?"},
]

In [52]:
response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        max_turns=3,
        temperature=0.75,
        tools=[orariov2, meteo]
    )
print(response.choices[0].message.content)

Il risultato di \( 11723,312 \times 18,214 \) è \( 213,554,730,688 \).


In [53]:
import math

def calcolatrice(a: float, b: float, operazione: str) -> dict:
    """Esegue una singola operazione aritmetica tra due numeri.

    Operazioni consentite:
    - somma: a + b
    - sottrazione: a - b
    - moltiplicazione: a * b
    - divisione: a / b

    Args:
        a: Primo numero.
        b: Secondo numero.
        operazione: Una tra 'somma', 'sottrazione',
            'moltiplicazione', 'divisione'.

    Returns:
        Un dizionario con operazione, input e risultato oppure un errore.
    """
    operazioni = {
        "somma": lambda x, y: x + y,
        "sottrazione": lambda x, y: x - y,
        "moltiplicazione": lambda x, y: x * y,
        "divisione": lambda x, y: x / y,
    }

    if operazione not in operazioni:
        return {
            "errore": "Operazione non consentita. Usa somma, sottrazione, moltiplicazione o divisione."
        }

    if operazione == "divisione" and b == 0:
        return {"errore": "Divisione per zero non consentita."}

    try:
        risultato = operazioni[operazione](a, b)
    except (TypeError, ValueError, OverflowError) as exc:
        return {"errore": f"Input non valido: {exc}"}

    if not math.isfinite(risultato):
        return {"errore": "Il risultato non è un numero finito."}

    return {
        "operazione": operazione,
        "a": a,
        "b": b,
        "risultato": risultato
    }

In [57]:
messages = [
    {"role": "system", "content": "Rispondi come un assistente generico.Se ti viene chiesta l'ora utiliza il tool orario. se ti viene chiesto quanti giorni mancano aduna certa data, utilizza come orario di partenza quello fornito dal tool orariov2. nel contesto del meteo per le domande relative alla nuvolositá, non usare le percentuali ma un'espressione piú popolare. se ti vengono chieste operazioni matematiche utilizza il tool calcolatrice. utilizza solo le operazioni, altrimenti di che non ce la fai."},
    {"role": "user", "content": "quanto fa 11723,312 x 18,214?"},
]

In [58]:
response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        max_turns=3,
        temperature=0.75,
        tools=[orariov2, meteo, calcolatrice]
    )
print(response.choices[0].message.content)

11723,312 moltiplicato per 18,214 fa circa 213528,40.


In [54]:
calcolatrice(10,5, "moltiplicazione")

{'operazione': 'moltiplicazione', 'a': 10, 'b': 5, 'risultato': 50}

In [59]:
choice = response.choices[0]
intermediate = getattr(choice, "intermediate_messages", None)

if intermediate is None:
    print("La risposta non espone intermediate_messages in questa versione/provider.")
else:
    print("Messaggi nella cronologia intermedia:", len(intermediate))
    for i, message in enumerate(intermediate):
        # Mostriamo solo ruolo e una piccola anteprima, non l'intero payload.
        role = message.get("role", "?") if isinstance(message, dict) else getattr(message, "role", "?")
        print(f"{i + 1}. role={role}")

Messaggi nella cronologia intermedia: 3
1. role=assistant
2. role=tool
3. role=assistant
